# Part 1: Student Data Exploration, Validation & Preprocessing Pipeline
### Project: AI-Powered Agentic Smart Campus Intelligence & Student Success Platform
**Author:** Sumith U (SRN: R24EA807) | 7th Sem B.Tech CSE (AI & ML), REVA University  
**Dataset:** Predict Students' Dropout and Academic Success (UCI Machine Learning Repository, ID: 697)  

---

## 1. Objective of Part 1
The goal of this notebook is to collect, explore, validate, and preprocess our student dataset.  
In this step, we **do not train machine learning models**. Instead, we build a trustworthy, clean dataset with zero data leakage that will feed into Part 2 for model training.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
base_dir = Path.cwd().parent
sys.path.insert(0, str(base_dir))

from src.data_loader import load_student_data, get_dataset_summary
from src.data_validation import DataValidator
from src.preprocessing import run_preprocessing_pipeline, build_preprocessor_pipeline

# Configure plotting style
sns.set_theme(style="whitegrid")
%matplotlib inline
print("Libraries and project modules imported successfully.")


## 2. Dataset Loading & Schema Inspection
We load the untouched raw CSV file (`data/raw/data.csv`) using our custom modular loader.  
Let us inspect the dimensions, column names, and the first five student records.


In [ ]:
df = load_student_data()
summary = get_dataset_summary(df)

print(f"Total Student Records: {summary['num_rows']}")
print(f"Total Attributes:      {summary['num_cols']}")
print(f"Total Missing Cells:   {summary['missing_values_total']}")
print(f"Total Duplicate Rows:  {summary['duplicate_rows_total']}")

# Preview first 5 rows
df.head()


## 3. Target Class Distribution
The target variable is **`Target`**, representing three potential academic outcomes:
1. **Graduate**: Successful degree completion (~49.9%)
2. **Dropout**: High-risk student attrition (~32.1%)
3. **Enrolled**: Active student in progress (~18.0%)


In [ ]:
counts = df['Target'].value_counts()
pcts = (df['Target'].value_counts(normalize=True) * 100).round(2)

target_df = pd.DataFrame({'Student Count': counts, 'Percentage (%)': pcts})
print(target_df)

# Visualize target distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
palette = {'Graduate': '#10B981', 'Dropout': '#F43F5E', 'Enrolled': '#F59E0B'}

sns.barplot(x=counts.index, y=counts.values, palette=palette, ax=ax1, edgecolor='black')
ax1.set_title('Target Class Distribution', fontsize=12, fontweight='bold')
ax1.set_ylabel('Student Count')

ax2.pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=[palette[k] for k in counts.index], startangle=140)
ax2.set_title('Target Proportion (%)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Automated Data Validation & Quality Audit
Before preprocessing, we verify that:
* No missing null values exist
* No duplicate student rows exist
* Ages are within realistic university enrollment bounds (17 - 70)
* Grades are within standard scale bounds


In [ ]:
validator = DataValidator(df)
validation_results = validator.run_all_validations()

for check, res in validation_results.items():
    if check != 'overall_passed':
        status = 'PASS' if res.get('passed', True) else 'FAIL'
        print(f"[{status}] {check}")

print(f"\nOverall Dataset Integrity Verified: {validation_results['overall_passed']}")


## 5. Feature Correlation Analysis
Let us examine the Pearson correlation between academic performance features and macroeconomic indicators.


In [ ]:
corr_cols = [
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Admission grade',
    'Previous qualification (grade)',
    'Age at enrollment',
    'Unemployment rate',
    'Inflation rate',
    'GDP'
]

plt.figure(figsize=(10, 8))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix: Academic & Economic Indicators', fontsize=12, fontweight='bold')
plt.show()


## 6. Anti-Leakage Preprocessing Pipeline
We execute our preprocessing pipeline:
1. Split into 80% train and 20% test using **stratification on Target**.
2. Fit `ColumnTransformer` (StandardScaler for numericals, OneHotEncoder for categoricals) **strictly on `X_train`**.
3. Transform `X_train` and `X_test`.
4. Export preprocessed matrices for Part 2.


In [ ]:
metadata = run_preprocessing_pipeline(test_size=0.20, random_state=42)

print("=== PREPROCESSING COMPLETED ===")
print(f"Training Records:   {metadata['train_records']}")
print(f"Testing Records:    {metadata['test_records']}")
print(f"Input Features:     {metadata['raw_features_count']} raw -> {metadata['processed_features_count']} one-hot encoded")
print(f"Train 3-Class Dist: {metadata['class_counts_train_3class']}")
print(f"Test 3-Class Dist:  {metadata['class_counts_test_3class']}")
print(f"Binary Dropout %:   Train={metadata['binary_dropout_train_pct']}%, Test={metadata['binary_dropout_test_pct']}%")


## 7. Summary & Next Steps (Ready for Part 2)
The dataset is now thoroughly audited and preprocessed. All outputs are saved inside:
* `data/processed/X_train.csv` & `data/processed/X_test.csv`
* `data/processed/y_train_3class.csv` & `data/processed/y_test_3class.csv` (Experiment A)
* `data/processed/y_train_binary.csv` & `data/processed/y_test_binary.csv` (Experiment B)
* `data/processed/preprocessor.joblib` (Fitted Scikit-Learn ColumnTransformer)

**We are now ready to proceed to Part 2: Prebuilt Machine Learning Model Training (Logistic Regression, Random Forest, CatBoost, and XGBoost).**
